In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
%matplotlib inline

In [3]:
raw_data = {
    "athlete_id": [
        "IND_101",
        "IND_102",
        "IND_103",
        "IND_104",
        "IND_105",
        "IND_106",
        "IND_107",
        "IND_108",
        "IND_109",
        "IND_110",
        "IND_111",
        "IND_112",
    ],
    "athlete_name": [
        "Manpreet S.",
        "PR Sreejesh",
        "Vinesh P.",
        "Mirabai C.",
        "Neeraj C.",
        "Bajrang P.",
        "Lovlina B.",
        "Savitra P.",
        "Balbir S. Sr.",
        "Dhyan C.",
        "KD Jadhav",
        "Karnam M.",
    ],
    "era": [
        "Modern",
        "Modern",
        "Modern",
        "Modern",
        "Modern",
        "Modern",
        "Modern",
        "Modern",
        "Golden",
        "Golden",
        "Golden",
        "Post-1990",
    ],
    "sport": [
        "Hockey",
        "hockey",
        "Wrestling",
        "Weightlifting",
        "Athletics",
        "Wrestling",
        "Boxing",
        "Hockey",
        "Hockey",
        "Hockey",
        "Wrestling",
        "Weightlifting",
    ],
    "gender": ["M", "M", "F", "F", "M", "M", "F", "F", "M", "M", "M", "F"],
    "height_cm": [
        173,
        183,
        160,
        150,
        182,
        166,
        177,
        165,
        178,
        170,
        165,
        np.nan,
    ],
    "weight_kg": [
        "70 kg",
        "85",
        "53 kg",
        "49",
        "86",
        "65 kg",
        "69",
        "58",
        "72",
        "64",
        "57",
        "54 kg",
    ],
    "body_fat_pct": [
        10.5,
        13.2,
        16.8,
        14.2,
        11.0,
        9.8,
        15.5,
        np.nan,
        np.nan,
        np.nan,
        np.nan,
        18.0,
    ],
    "daily_caloric_intake": [
        3800,
        4100,
        2800,
        2500,
        4500,
        3400,
        3100,
        3200,
        np.nan,
        np.nan,
        np.nan,
        2900,
    ],
    "medal_won": [
        "Bronze",
        "Bronze",
        "None",
        "Silver",
        "Gold",
        "Bronze",
        "Bronze",
        "None",
        "Gold",
        "Gold",
        "Bronze",
        "Bronze",
    ],
}

df = pd.DataFrame(raw_data)

Type Sanitization & Cleaning:

Notice sport has inconsistent casing ("Hockey" vs "hockey"). Standardize the column to title-case.

weight_kg contains strings with " kg" attached to some rows. Strip non-numeric tokens and cast the entire column to float64.

In [4]:
df["sport"].str.lower() # to convert all of them into lower case
df["sport"] = df["sport"].str.title()
print(df["sport"]) # the first letter will be capital all other remains same

0            Hockey
1            Hockey
2         Wrestling
3     Weightlifting
4         Athletics
5         Wrestling
6            Boxing
7            Hockey
8            Hockey
9            Hockey
10        Wrestling
11    Weightlifting
Name: sport, dtype: str


In [5]:
# df['column'] = df['column'].str.replace(r'\D', '', regex=True) # this preserve only integers
#df['column'] = df['column'].str.replace(r'[^0-9.]', '', regex=True) # this preserve decimal also
df["weight_kg"] = df["weight_kg"].str.replace(r'[^0-9]','',regex=True)
print(df["weight_kg"])

0     70
1     85
2     53
3     49
4     86
5     65
6     69
7     58
8     72
9     64
10    57
11    54
Name: weight_kg, dtype: str


In [6]:
#df['column'] = pd.to_numeric(df['column'], errors='coerce') this change invalid number to nan
#df['column'] = df['column'].astype(dtype)
df["weight_kg"] = df["weight_kg"].astype(float)
print(df["weight_kg"])

0     70.0
1     85.0
2     53.0
3     49.0
4     86.0
5     65.0
6     69.0
7     58.0
8     72.0
9     64.0
10    57.0
11    54.0
Name: weight_kg, dtype: float64


Feature Engineering (Biometrics): <br> Compute BMI ($\text{BMI} = \frac{\text{weight in kg}}{(\text{height in m})^2}$).<br>Flag athletes who are in combat/weight-restricted categories (Wrestling, Boxing, Weightlifting) versus open-field team sports (Hockey, Athletics) under a new boolean column is_weight_class_sport.

In [7]:
df["height_cm"] = pd.to_numeric(df["height_cm"],errors='coerce').astype(float)
#df["height_cm"] = df["height_cm"].dropna() didnt work because of index miss alignment 
df.dropna(subset=["height_cm"], inplace=True)
print(df["height_cm"])

0     173.0
1     183.0
2     160.0
3     150.0
4     182.0
5     166.0
6     177.0
7     165.0
8     178.0
9     170.0
10    165.0
Name: height_cm, dtype: float64


In [8]:
## BMI calculation 
weight = df["weight_kg"].to_numpy()
height = df["height_cm"].to_numpy()
BMI = weight/height**2
BMI*=10000
print(BMI)

[23.38868656 25.38146854 20.703125   21.77777778 25.96304794 23.58832922
 22.02432251 21.30394858 22.72440348 22.14532872 20.93663912]


In [9]:
weight_class_sports = ["Wrestling", "Boxing", "Weightlifting"]

df["is_weight_class_sport"] = df["sport"].isin(weight_class_sports)

In [10]:
print(df[["sport","is_weight_class_sport"]].to_string(index=False))

        sport  is_weight_class_sport
       Hockey                  False
       Hockey                  False
    Wrestling                   True
Weightlifting                   True
    Athletics                  False
    Wrestling                   True
       Boxing                   True
       Hockey                  False
       Hockey                  False
       Hockey                  False
    Wrestling                   True


In [11]:
#df.loc[condition, ["column1", "column2", ...]]
df.loc[df["is_weight_class_sport"], ["sport", "weight_kg"]]

,sport,weight_kg
2,Wrestling,53.0
3,Weightlifting,49.0
5,Wrestling,65.0
6,Boxing,69.0
10,Wrestling,57.0


In [12]:
df

,athlete_id,athlete_name,era,sport,gender,height_cm,weight_kg,body_fat_pct,daily_caloric_intake,medal_won,is_weight_class_sport
0,IND_101,Manpreet S.,Modern,Hockey,M,173.0,70.0,10.5,3800.0,Bronze,False
1,IND_102,PR Sreejesh,Modern,Hockey,M,183.0,85.0,13.2,4100.0,Bronze,False
2,IND_103,Vinesh P.,Modern,Wrestling,F,160.0,53.0,16.8,2800.0,None,True
3,IND_104,Mirabai C.,Modern,Weightlifting,F,150.0,49.0,14.2,2500.0,Silver,True
4,IND_105,Neeraj C.,Modern,Athletics,M,182.0,86.0,11.0,4500.0,Gold,False
5,IND_106,Bajrang P.,Modern,Wrestling,M,166.0,65.0,9.8,3400.0,Bronze,True
6,IND_107,Lovlina B.,Modern,Boxing,F,177.0,69.0,15.5,3100.0,Bronze,True
7,IND_108,Savitra P.,Modern,Hockey,F,165.0,58.0,NaN,3200.0,None,False
8,IND_109,Balbir S. Sr.,Golden,Hockey,M,178.0,72.0,NaN,NaN,Gold,False
9,IND_110,Dhyan C.,Golden,Hockey,M,170.0,64.0,NaN,NaN,Gold,False


Multi-level Aggregations (groupby + agg):

Compute the median height_cm, mean weight_kg, and standard deviation of daily_caloric_intake across each unique combination of era and sport.

#grouping coloums together
#grp = df.groupby(["era","sport"])<br>
The grouping happens like this:<br>
Modern + Athletics<br>
Heights: 170, 180, 160<br>

Modern + Swimming<br>
Heights: 175, 185<br>

Ancient + Athletics<br>
Heights: 165<br>
This looks something like this

In [13]:
groups = df.groupby(["era","sport"])

## Tasks
0. Group by era, sport
1. calculate median height
2. mean weight
3. sd of calorie intake

In [14]:
groups[["height_cm"]].median()

groups[["weight_kg"]].mean()

groups["daily_caloric_intake"].std() # add dropna here to exclude nan

#df["daily_caloric_intake"].dropna() this didnt work 

era     sport        
Golden  Hockey                  NaN
        Wrestling               NaN
Modern  Athletics               NaN
        Boxing                  NaN
        Hockey           458.257569
        Weightlifting           NaN
        Wrestling        424.264069
Name: daily_caloric_intake, dtype: float64

## Understanding `groupby()` with `transform()` and `fillna()`

Suppose our DataFrame contains missing heights for both male and female athletes.

| Index | gender | sport | height_cm |
|---:|---|---|---:|
| 0 | Male | Hockey | 180 |
| 1 | Male | Hockey | NaN |
| 2 | Male | Hockey | 190 |
| 3 | Female | Hockey | NaN |
| 4 | Female | Hockey | 160 |
| 5 | Female | Hockey | NaN |

### Step 1: Calculate the group means

```python
group_means = df.groupby(
    ["gender", "sport"]
)["height_cm"].transform("mean")
```

Pandas groups the rows based on both `gender` and `sport`.

- **Male + Hockey:** Mean = (180 + 190) / 2 = 185
- **Female + Hockey:** Mean = 160 / 1 = 160

**Note:** `mean()` automatically ignores `NaN` values while calculating the mean.

### Step 2: Understand the output of `transform("mean")`

Unlike `groupby().mean()`, which returns one mean per group, `transform("mean")` returns a value for every original row.

| Index | gender | height_cm | group_means |
|---:|---|---:|---:|
| 0 | Male | 180 | 185 |
| 1 | Male | NaN | 185 |
| 2 | Male | 190 | 185 |
| 3 | Female | NaN | 160 |
| 4 | Female | 160 | 160 |
| 5 | Female | NaN | 160 |

Therefore, `group_means` contains:

```python
[185, 185, 185, 160, 160, 160]
```

The mean of each group is repeated for all rows belonging to that group. The original row count and index are preserved.

### Step 3: Fill the missing heights

```python
df["height_cm"] = df["height_cm"].fillna(group_means)
```

`fillna()` checks each row:

- If `height_cm` already has a value, it keeps the existing value.
- If `height_cm` is `NaN`, it replaces it with the corresponding value from `group_means`.

The final DataFrame becomes:

| Index | gender | sport | height_cm |
|---:|---|---|---:|
| 0 | Male | Hockey | 180 |
| 1 | Male | Hockey | 185 |
| 2 | Male | Hockey | 190 |
| 3 | Female | Hockey | 160 |
| 4 | Female | Hockey | 160 |
| 5 | Female | Hockey | 160 |

### Key takeaway

- `groupby()` divides the data into groups.
- `transform("mean")` calculates each group's mean and assigns it to every row in that group.
- `fillna()` replaces missing values using the corresponding group means.

**Edge case:** If every athlete in a particular gender–sport group has a missing height, that group's mean will also be `NaN`. Consequently, `fillna()` cannot impute those rows.

**Remember:** `transform()` is useful when you want to calculate a group-level statistic and use it alongside every original row.

In [15]:
df["height_cm"]

0     173.0
1     183.0
2     160.0
3     150.0
4     182.0
5     166.0
6     177.0
7     165.0
8     178.0
9     170.0
10    165.0
Name: height_cm, dtype: float64

In [16]:
df.iloc[0]
df.loc[len(df)] = [
    "IND_102", "New Athlete", "Modern", "Hockey",
    "M", float("nan"), 70.0, 10.5, 3800.0,
    "Bronze", False
]

In [17]:
df

,athlete_id,athlete_name,era,sport,gender,height_cm,weight_kg,body_fat_pct,daily_caloric_intake,medal_won,is_weight_class_sport
0,IND_101,Manpreet S.,Modern,Hockey,M,173.0,70.0,10.5,3800.0,Bronze,False
1,IND_102,PR Sreejesh,Modern,Hockey,M,183.0,85.0,13.2,4100.0,Bronze,False
2,IND_103,Vinesh P.,Modern,Wrestling,F,160.0,53.0,16.8,2800.0,None,True
3,IND_104,Mirabai C.,Modern,Weightlifting,F,150.0,49.0,14.2,2500.0,Silver,True
4,IND_105,Neeraj C.,Modern,Athletics,M,182.0,86.0,11.0,4500.0,Gold,False
5,IND_106,Bajrang P.,Modern,Wrestling,M,166.0,65.0,9.8,3400.0,Bronze,True
6,IND_107,Lovlina B.,Modern,Boxing,F,177.0,69.0,15.5,3100.0,Bronze,True
7,IND_108,Savitra P.,Modern,Hockey,F,165.0,58.0,NaN,3200.0,None,False
8,IND_109,Balbir S. Sr.,Golden,Hockey,M,178.0,72.0,NaN,NaN,Gold,False
9,IND_110,Dhyan C.,Golden,Hockey,M,170.0,64.0,NaN,NaN,Gold,False


### Tasks
1. Impute missing height_cm using the mean height of the athlete's corresponding gender and sport group, rather than the global mean.
2. Add a column relative_caloric_diff showing how much an athlete's calorie intake deviates from their respective sport's average calorie intake.

In [18]:
group_means = df.groupby(["gender","sport"])["height_cm"].transform("mean")
df["height_cm"] = df["height_cm"].fillna(group_means)
df

,athlete_id,athlete_name,era,sport,gender,height_cm,weight_kg,body_fat_pct,daily_caloric_intake,medal_won,is_weight_class_sport
0,IND_101,Manpreet S.,Modern,Hockey,M,173.0,70.0,10.5,3800.0,Bronze,False
1,IND_102,PR Sreejesh,Modern,Hockey,M,183.0,85.0,13.2,4100.0,Bronze,False
2,IND_103,Vinesh P.,Modern,Wrestling,F,160.0,53.0,16.8,2800.0,None,True
3,IND_104,Mirabai C.,Modern,Weightlifting,F,150.0,49.0,14.2,2500.0,Silver,True
4,IND_105,Neeraj C.,Modern,Athletics,M,182.0,86.0,11.0,4500.0,Gold,False
5,IND_106,Bajrang P.,Modern,Wrestling,M,166.0,65.0,9.8,3400.0,Bronze,True
6,IND_107,Lovlina B.,Modern,Boxing,F,177.0,69.0,15.5,3100.0,Bronze,True
7,IND_108,Savitra P.,Modern,Hockey,F,165.0,58.0,NaN,3200.0,None,False
8,IND_109,Balbir S. Sr.,Golden,Hockey,M,178.0,72.0,NaN,NaN,Gold,False
9,IND_110,Dhyan C.,Golden,Hockey,M,170.0,64.0,NaN,NaN,Gold,False


In [19]:
grp_mean = df["daily_caloric_intake"].mean()
df["daily_caloric_intake"] = df["daily_caloric_intake"].fillna(grp_mean)
df["rel_caloric_intake"] = df["daily_caloric_intake"] - grp_mean
df["rel_caloric_intake"]

0      333.333333
1      633.333333
2     -666.666667
3     -966.666667
4     1033.333333
5      -66.666667
6     -366.666667
7     -266.666667
8        0.000000
9        0.000000
10       0.000000
11     333.333333
Name: rel_caloric_intake, dtype: float64

### Task
1. Era Comparison:

Filter athletes with a medal (medal_won != "None") and contrast the height/weight profiles of Modern vs Golden-era athletes.

In [21]:
medals_df = df[df["medal_won"]!="None"]

In [22]:
era_comparison = (
    medals_df.groupby("era")[["height_cm", "weight_kg"]]
    .agg(["count", "mean", "median", "std"])
    .round(2)
)

print(era_comparison)

       height_cm                       weight_kg                     
           count    mean median    std     count   mean median    std
era                                                                  
Golden         3  171.00  170.0   6.56         3  64.33   64.0   7.51
Modern         7  172.43  176.0  11.41         7  70.57   70.0  12.55


In [24]:
era_comparison.style.format("{:.3f}").set_caption("Athlete Measurements by Era")